# Задание 4: отдельное исследование предобработки

Рабочая копия исходного `task4.ipynb`; исходный файл остаётся неизменным. Используются исключительно предоставленные 14000 размеченных строк. Модель и метрики реализованы в PyTorch. Ансамбли, внешние данные и использование `id` как признака исключены.

Проверяется влияние преобразований признаков на одну фиксированную residual-сеть. Основное разбиение полностью совпадает с baseline: 9800 train / 2100 validation / 2100 audit. Audit не оценивается; лучшие идеи проверяются на двух дополнительных разбиениях внутри 11900 development-строк. Каждая метрика ниже — MSE в годах², если явно не указано RMSE. Все статистики преобразований обучаются только на train соответствующего разбиения.

Принципы обучения взяты из локального `notebooks/pytorch_intro.ipynb`: `Dataset` / `DataLoader`, `nn.Module`, autograd, `zero_grad → backward → step`, `eval` и `inference_mode`.


In [1]:
import base64
import copy
import hashlib
import io
import math
import random
import time
import unittest
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SELECTION_METRIC = "rmse"  # "rmse", "mse" или "mae"; см. пояснение выше.
assert SELECTION_METRIC in {"rmse", "mse", "mae"}
MAX_EPOCHS = 160
PATIENCE = 25
BATCH_SIZE = 256
# CPU выбран для воспроизводимого сравнения двух реализаций оптимизатора.
# Для большого запуска можно указать "cuda" или "mps", если устройство доступно.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)


def seed_everything(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)


seed_everything()
print(f"torch={torch.__version__}; numpy={np.__version__}; "
      f"pandas={pd.__version__}; sklearn={sklearn.__version__}; device={DEVICE}")
import json, pickle, statistics
from sklearn.decomposition import PCA
ARTIFACT_DIR = Path("/tmp/task4_preprocessing")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


torch=2.14.1; numpy=2.4.6; pandas=3.0.3; sklearn=1.9.0; device=cpu


In [2]:
candidates = [Path("task4_data"), Path("homeworks/task4_data")]
DATA_DIR = next((p for p in candidates if (p / "train_x.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Запустите ноутбук из корня репозитория или из homeworks.")

train_x = pd.read_csv(DATA_DIR / "train_x.csv", index_col=0)
train_y = pd.read_csv(DATA_DIR / "train_y.csv", index_col=0)
test_table = pd.read_csv(DATA_DIR / "test_x.csv")
assert train_x.index.is_unique and train_y.index.is_unique
assert set(train_x.index) == set(train_y.index), "Индексы признаков и меток не совпадают"
train_y = train_y.loc[train_x.index]
assert list(train_y.columns) == ["year"]
assert "id" in test_table and test_table["id"].is_unique
assert not test_table["id"].isna().any()

feature_columns = list(train_x.columns)
assert len(feature_columns) == 90
assert set(test_table.columns) == set(feature_columns) | {"id"}
test_ids = test_table["id"].copy()
test_x = test_table.loc[:, feature_columns].copy()
assert all(pd.api.types.is_numeric_dtype(t) for t in train_x.dtypes)
assert all(pd.api.types.is_numeric_dtype(t) for t in test_x.dtypes)
X = train_x.to_numpy(dtype=np.float64)
X_test = test_x.to_numpy(dtype=np.float64)
y = torch.tensor(train_y["year"].to_numpy(dtype=np.float32))
assert torch.isfinite(y).all() and len(y) == len(X)
assert torch.equal(y, y.round()), "Годы должны быть целыми"
assert not np.isinf(X).any() and not np.isinf(X_test).any()

print(f"Обучение: {X.shape}; тест: {X_test.shape}; целевая переменная: year")
print(f"Пропуски: train={np.isnan(X).sum()}, test={np.isnan(X_test).sum()}")
print(f"Точные дубликаты train: {train_x.duplicated().sum()}")
for name in ("train_x.csv", "train_y.csv", "test_x.csv"):
    print(name, "SHA256:", hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest())

Обучение: (14000, 90); тест: (6000, 90); целевая переменная: year
Пропуски: train=0, test=0
Точные дубликаты train: 0
train_x.csv SHA256: aab2f64a4fdb0fd6af98ed95fedc9a3d9a54506b1f607da725926f654f5bff89
train_y.csv SHA256: 1f72519091e6e08948c430a1db9b181df28095e0cc9f29694570b40daed683bf
test_x.csv SHA256: 54f91d13c6f61fd634bae76bdad109471585a4ce0b308159be16f768134b4046


In [3]:
# NumPy/sklearn здесь применяются только к предварительному разбиению данных.
periods = np.digitize(y.tolist(), bins=[1970, 1980, 1990, 2000, 2005])
indices = np.arange(len(X))
development_idx, audit_idx = train_test_split(
    indices, test_size=0.15, random_state=SEED, stratify=periods
)
train_idx, valid_idx = train_test_split(
    development_idx, test_size=0.15 / 0.85, random_state=SEED,
    stratify=periods[development_idx]
)
assert not (set(train_idx) & set(valid_idx) or set(train_idx) & set(audit_idx)
            or set(valid_idx) & set(audit_idx))
assert len(set(train_idx) | set(valid_idx) | set(audit_idx)) == len(X)
row_hashes = pd.util.hash_pandas_object(train_x, index=False)
assert not (set(row_hashes.iloc[train_idx]) & set(row_hashes.iloc[valid_idx]))
assert not (set(row_hashes.iloc[development_idx]) & set(row_hashes.iloc[audit_idx]))
y_train, y_valid = y[train_idx], y[valid_idx]
print(f"train={len(train_idx)}, validation={len(valid_idx)}, audit={len(audit_idx)}")
print(f"Train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, "
      f"min={y_train.min():.0f}, max={y_train.max():.0f}")

# audit_idx зафиксирован; ни предсказания, ни метрики audit здесь не вычисляются.


train=9800, validation=2100, audit=2100
Train: mean=1998.37, std=11.07, min=1922, max=2011


## Предобработка и контроль утечки

Сравниваются: ограничение выбросов, нелинейное сжатие всех или отдельных групп признаков, квантильные преобразования, PCA с частичным отбеливанием, добавление масштаба признаков каждой строки. Первые 12 столбцов описывают средние, следующие 78 — ковариации; порядок отдельных ковариаций неизвестен, поэтому восстановление матрицы ковариации не применяется.

`whiten_power=0` означает только поворот PCA; `0.5` — частичное и `1` — полное отбеливание. Усиление малых компонент ограничено снизу долей 0.01 от максимальной дисперсии. При добавлении row-scale признаков вычисляются только четыре агрегата строки: RMS и среднее абсолютное значение для двух групп.


In [4]:
class SearchPreprocessor:
    def __init__(self, spec):
        self.spec = dict(spec)
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)

    def _mask(self, d):
        block = self.spec.get("block", "all")
        return slice(0, 12) if block == "means" else (slice(12, d) if block == "cov" else slice(0, d))

    def fit(self, x):
        z = self.imputer.fit_transform(x)
        self.d = z.shape[1]
        kind = self.spec["kind"]
        self.robust = RobustScaler().fit(z)
        if kind == "winsor":
            q = self.spec["q"]
            self.bounds = np.quantile(z, [q, 1-q], axis=0)
        if kind in {"quantile", "hybrid"}:
            self.qt = QuantileTransformer(n_quantiles=min(self.spec.get("n_quantiles", 512), len(z)),
                        output_distribution=self.spec.get("distribution", "normal"),
                        subsample=None, random_state=42).fit(z[:, self._mask(self.d)])
        z = self._nonlinear(z)
        self.scaler = StandardScaler().fit(z)
        if kind == "pca":
            zz = self.scaler.transform(z)
            self.pca = PCA(n_components=self.spec.get("components", self.d), svd_solver="full").fit(zz)
            self.pca_scale = np.maximum(self.pca.explained_variance_, self.pca.explained_variance_.max()*0.01) ** (self.spec.get("whiten_power", 0)/2)
        if kind == "augment":
            self.aug_scaler = StandardScaler().fit(self._augment(self.scaler.transform(z)))
        return self

    def _nonlinear(self, z):
        z = z.copy()
        kind = self.spec["kind"]
        mask = self._mask(z.shape[1])
        if kind == "winsor":
            return np.clip(z, self.bounds[0], self.bounds[1])
        if kind in {"asinh", "signedlog", "robust"}:
            r = self.robust.transform(z)
            if kind == "robust": return r
            v = r[:, mask] / self.spec.get("scale", 1.)
            z[:, mask] = np.arcsinh(v) if kind == "asinh" else np.sign(v)*np.log1p(np.abs(v))
        if kind in {"quantile", "hybrid"}:
            z[:, mask] = self.qt.transform(z[:, mask])
        return z

    @staticmethod
    def _augment(z):
        extra = []
        for a in (z[:, :12], z[:, 12:]):
            extra.extend([np.log1p(np.sqrt(np.mean(a*a, axis=1))), np.log1p(np.mean(np.abs(a), axis=1))])
        return np.stack(extra, axis=1)

    def transform(self, x):
        z = self._nonlinear(self.imputer.transform(x))
        if self.spec["kind"] != "robust": z = self.scaler.transform(z)
        if self.spec["kind"] == "clip": z = np.clip(z, -self.spec["limit"], self.spec["limit"])
        if self.spec["kind"] == "pca": z = self.pca.transform(z)/self.pca_scale
        if self.spec["kind"] == "augment": z = np.concatenate([z, self.aug_scaler.transform(self._augment(z))], axis=1)
        assert np.isfinite(z).all()
        return torch.from_numpy(z.astype(np.float32))


In [5]:
class TargetScaler:
    def __init__(self, target):
        self.mean = float(target.mean())
        self.std = max(float(target.std(unbiased=False)), 1e-6)

    def encode(self, target):
        return (target - self.mean) / self.std

    def decode(self, prediction):
        return prediction * self.std + self.mean


@torch.inference_mode()
def regression_metrics(prediction, target):
    assert prediction.shape == target.shape
    error = prediction.double() - target.double()
    mse = error.square().mean().item()
    return {"rmse": math.sqrt(mse), "mse": mse, "mae": error.abs().mean().item()}


def print_metrics(label, values):
    print(f"{label}: RMSE={values['rmse']:.4f}; MSE={values['mse']:.4f}; MAE={values['mae']:.4f}")




In [6]:
class CustomAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)
        for group in self.param_groups:
            if not math.isfinite(group["lr"]) or group["lr"] < 0:
                raise ValueError("lr должен быть конечным и неотрицательным")
            if not math.isfinite(group["eps"]) or group["eps"] <= 0:
                raise ValueError("eps должен быть конечным и положительным")
            if not math.isfinite(group["weight_decay"]) or group["weight_decay"] < 0:
                raise ValueError("weight_decay должен быть конечным и неотрицательным")
            if len(group["betas"]) != 2 or not all(0 <= b < 1 for b in group["betas"]):
                raise ValueError("betas должны лежать в [0, 1)")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for parameter in group["params"]:
                grad = parameter.grad
                if grad is None:
                    continue
                if grad.is_sparse or torch.is_complex(parameter):
                    raise RuntimeError("Поддерживаются только плотные вещественные градиенты")
                state = self.state[parameter]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(parameter)
                    state["exp_avg_sq"] = torch.zeros_like(parameter)
                state["step"] += 1
                m, v = state["exp_avg"], state["exp_avg_sq"]
                m.lerp_(grad, 1 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                correction1 = 1 - beta1 ** state["step"]
                correction2 = 1 - beta2 ** state["step"]
                denominator = v.sqrt().div_(math.sqrt(correction2)).add_(group["eps"])
                parameter.mul_(1 - group["lr"] * group["weight_decay"])
                parameter.addcdiv_(m, denominator, value=-group["lr"] / correction1)
        return loss

In [7]:
class TestCustomAdamW(unittest.TestCase):
    def test_reference_steps_and_parameter_groups(self):
        generator = torch.Generator().manual_seed(101)
        ours = [nn.Parameter(torch.randn(3, 4, generator=generator, dtype=torch.float64)),
                nn.Parameter(torch.randn(4, generator=generator, dtype=torch.float64))]
        reference = [nn.Parameter(p.detach().clone()) for p in ours]
        def groups(parameters):
            return [{"params": [parameters[0]], "lr": 0.003, "weight_decay": 0.1},
                    {"params": [parameters[1]], "lr": 0.002, "weight_decay": 0.0}]
        custom = CustomAdamW(groups(ours), betas=(0.8, 0.95), eps=1e-6)
        builtin = torch.optim.AdamW(groups(reference), betas=(0.8, 0.95), eps=1e-6,
                                    foreach=False, fused=False)
        for step in range(20):
            for i, (p, q) in enumerate(zip(ours, reference)):
                grad = torch.randn(p.shape, generator=generator, dtype=p.dtype)
                if step % 4 == 0 and i == 1:
                    p.grad = q.grad = None
                else:
                    p.grad = torch.zeros_like(grad) if step == 7 else grad
                    q.grad = p.grad.clone()
            custom.step()
            builtin.step()
            for p, q in zip(ours, reference):
                torch.testing.assert_close(p, q, rtol=1e-11, atol=1e-12)

    def test_none_vs_zero_gradient(self):
        p = nn.Parameter(torch.tensor([2.0], dtype=torch.float64))
        optimizer = CustomAdamW([p], lr=0.1, weight_decay=0.2)
        optimizer.step()
        self.assertEqual(p.item(), 2.0)
        self.assertNotIn(p, optimizer.state)
        p.grad = torch.zeros_like(p)
        optimizer.step()
        self.assertAlmostEqual(p.item(), 1.96)
        self.assertEqual(optimizer.state[p]["step"], 1)

    def test_restore_state(self):
        p = nn.Parameter(torch.tensor([1.0, -2.0], dtype=torch.float64))
        a = CustomAdamW([p])
        p.grad = torch.tensor([0.3, -0.1], dtype=p.dtype)
        a.step()
        q = nn.Parameter(p.detach().clone())
        b = CustomAdamW([q])
        b.load_state_dict(copy.deepcopy(a.state_dict()))
        for _ in range(5):
            p.grad = torch.tensor([-0.2, 0.4], dtype=p.dtype)
            q.grad = p.grad.clone()
            a.step()
            b.step()
        torch.testing.assert_close(p, q, rtol=0, atol=0)

    def test_invalid_settings(self):
        for kwargs in [{"lr": -1}, {"eps": 0}, {"betas": (1, 0.9)}, {"weight_decay": -1}]:
            with self.assertRaises(ValueError):
                CustomAdamW([nn.Parameter(torch.ones(1))], **kwargs)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCustomAdamW)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_invalid_settings (__main__.TestCustomAdamW.test_invalid_settings) ... ok
test_none_vs_zero_gradient (__main__.TestCustomAdamW.test_none_vs_zero_gradient) ... ok
test_reference_steps_and_parameter_groups (__main__.TestCustomAdamW.test_reference_steps_and_parameter_groups) ... ok
test_restore_state (__main__.TestCustomAdamW.test_restore_state) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.014s

OK


In [8]:
class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(width), nn.Linear(width, width * 2), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(width * 2, width), nn.Dropout(dropout)
        )
        nn.init.zeros_(self.net[-2].weight)
        nn.init.zeros_(self.net[-2].bias)

    def forward(self, x):
        return x + self.net(x)


class YearRegressor(nn.Module):
    def __init__(self, n_features, architecture="residual", width=128, dropout=0.15):
        super().__init__()
        if architecture == "linear":
            self.net = nn.Linear(n_features, 1)
        elif architecture == "mlp":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1)
            )
        elif architecture == "residual":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.SiLU(),
                ResidualBlock(width, dropout), ResidualBlock(width, dropout),
                nn.LayerNorm(width), nn.SiLU(), nn.Linear(width, 1)
            )
        else:
            raise ValueError(architecture)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def build_optimizer(model, kind, lr, weight_decay):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    if kind == "custom":
        return CustomAdamW(groups, lr=lr)
    if kind == "builtin":
        return torch.optim.AdamW(groups, lr=lr, foreach=False, fused=False)
    raise ValueError(kind)


@torch.inference_mode()
def predict(model, features, target_scaler, batch_size=1024):
    model.eval()
    outputs = [model(batch.to(DEVICE)).cpu() for batch in features.split(batch_size)]
    return target_scaler.decode(torch.cat(outputs))


def make_loader(features, targets, seed):
    return DataLoader(TensorDataset(features, targets), batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)


def train_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.L1Loss() if SELECTION_METRIC == "mae" else nn.MSELoss()
    total_loss, total_grad, count = 0.0, 0.0, 0
    for features, targets in loader:
        features, targets = features.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(features)
        assert prediction.shape == targets.shape
        loss = loss_fn(prediction, targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Обучение разошлось")
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0,
                                           error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss.item() * len(features)
        total_grad += float(grad_norm) * len(features)
        count += len(features)
    return total_loss / count, total_grad / count


def learning_rate(config, epoch):
    # Та же функция используется при final refit, без изменения горизонта.
    phase = (epoch - 1) / max(1, MAX_EPOCHS - 1)
    return config["lr"] * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * phase)))




## Контролируемый поиск

Основная сеть, как в baseline: ширина 128, два residual-блока, LayerNorm, SiLU, dropout 0.15, AdamW (`lr=0.001`, `weight_decay=0.01`), batch 256, MSE стандартизированного года. Проверяются разные представления входа; архитектура удерживается фиксированной. Горизонт cosine learning-rate всегда 160 эпох, включая финальное обучение. Поиск ограничен 80 эпохами с patience 15; baseline и подтверждение — 160 эпохами с patience 25. Изменение patience не изменяет траекторию ранних эпох.

Для каждого запуска сохраняются config, вся кривая validation, лучшая эпоха, state_dict, обученная предобработка и предсказания validation вместе с исходными позиционными индексами. Это позволяет воспроизвести оценку и сравнить ошибки одинаковых объектов.


In [9]:
def run_config(config, split_seed=42, max_epochs=80, patience=15, optimizer_kind="builtin"):
    ti, vi = train_test_split(development_idx, test_size=0.15/0.85, random_state=split_seed,
                             stratify=periods[development_idx])
    seed_everything(split_seed)
    pp = SearchPreprocessor(config["preprocessing"]).fit(X[ti])
    xt, xv = pp.transform(X[ti]), pp.transform(X[vi])
    scaler = TargetScaler(y[ti])
    loader = make_loader(xt, scaler.encode(y[ti]), split_seed)
    model = YearRegressor(xt.shape[1], config["architecture"], config["width"], config["dropout"])
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best_mse, stale, history, best_epoch, best_state = math.inf, 0, [], 0, None
    started = time.perf_counter()
    for epoch in range(1, max_epochs+1):
        for group in optimizer.param_groups: group["lr"] = learning_rate(config, epoch)
        loss, grad = train_epoch(model, loader, optimizer)
        pred = predict(model, xv, scaler)
        metrics = regression_metrics(pred, y[vi])
        history.append(dict(epoch=epoch, train_loss=loss, grad_norm=grad, **metrics))
        if metrics["mse"] < best_mse:
            best_mse, best_epoch, stale = metrics["mse"], epoch, 0
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: stale += 1
        if stale >= patience: break
    model.load_state_dict(best_state)
    pred = predict(model, xv, scaler)
    metrics = regression_metrics(pred, y[vi])
    rid = f"{config['id']}_s{split_seed}_{optimizer_kind}"
    result = dict(id=rid, config=config, split_seed=split_seed, best_epoch=best_epoch,
                  metrics=metrics, history=history, seconds=time.perf_counter()-started,
                  optimizer=optimizer_kind, checkpoint=str(ARTIFACT_DIR/f"{rid}.pt"))
    checkpoint = dict(state_dict=best_state, config=config, split_seed=split_seed,
                      target_mean=scaler.mean, target_std=scaler.std, train_idx=torch.tensor(ti),
                      valid_idx=torch.tensor(vi), valid_prediction=pred, result=result)
    torch.save(checkpoint, ARTIFACT_DIR/f"{rid}.pt")
    with open(ARTIFACT_DIR/f"{rid}_preprocessing.pkl", "wb") as f: pickle.dump(pp, f)
    print(f"{rid:42s} MSE={metrics['mse']:.6f} RMSE={metrics['rmse']:.6f} epoch={best_epoch:3d} seconds={result['seconds']:.1f}", flush=True)
    return result


def persist_results(results):
    Path("/tmp/task4_preprocessing_results.json").write_text(json.dumps(results, indent=2))

common = dict(architecture="residual", width=128, dropout=0.15, lr=0.001, weight_decay=0.01)
configs = []
def add_config(cid, spec, **kwargs):
    configs.append(dict(common, id=cid, preprocessing=spec, **kwargs))
add_config("standard", dict(kind="standard"))
add_config("robust", dict(kind="robust"))
for limit in [2., 3., 5., 10.]: add_config(f"clip{limit:g}", dict(kind="clip", limit=limit))
for q in [.001, .005, .01]: add_config(f"winsor{q:g}", dict(kind="winsor", q=q))
for scale in [.25, 1., 4.]: add_config(f"asinh{scale:g}", dict(kind="asinh", scale=scale))
for block in ["means", "cov"]:
    add_config(f"asinh_{block}", dict(kind="asinh", block=block))
for block in ["all", "means", "cov"]:
    add_config(f"log_{block}", dict(kind="signedlog", block=block))
for nq in [128,512,1024]: add_config(f"quantile{nq}", dict(kind="quantile", n_quantiles=nq))
add_config("quantile_uniform", dict(kind="quantile", distribution="uniform"))
for block in ["means", "cov"]: add_config(f"quantile_{block}", dict(kind="hybrid", block=block))
for power in [0., .25, .5, 1.]: add_config(f"pca{power:g}", dict(kind="pca", whiten_power=power))
for ncomp in [50,70]: add_config(f"pca{ncomp}_partial", dict(kind="pca", components=ncomp, whiten_power=.5))
add_config("rowscale", dict(kind="augment"))
assert len(configs) >= 25
print(f"Конфигураций предобработки: {len(configs)}")
results=[]
for config in configs:
    result=run_config(config, max_epochs=160 if config['id']=='standard' else 80,
                      patience=25 if config['id']=='standard' else 15)
    results.append(result)
    persist_results(results)
print("\nОсновное разбиение, лучшие варианты:")
for r in sorted(results,key=lambda r:r['metrics']['mse'])[:12]:
    print(r['config']['id'], r['metrics'], 'epoch',r['best_epoch'])


Конфигураций предобработки: 30
standard_s42_builtin                       MSE=83.889076 RMSE=9.159098 epoch=  4 seconds=2.6
robust_s42_builtin                         MSE=85.772892 RMSE=9.261366 epoch=  2 seconds=1.6
clip2_s42_builtin                          MSE=83.997404 RMSE=9.165010 epoch=  4 seconds=1.9
clip3_s42_builtin                          MSE=83.860158 RMSE=9.157519 epoch=  4 seconds=1.9
clip5_s42_builtin                          MSE=83.883188 RMSE=9.158777 epoch=  4 seconds=1.8
clip10_s42_builtin                         MSE=83.914907 RMSE=9.160508 epoch=  4 seconds=1.8
winsor0.001_s42_builtin                    MSE=84.032975 RMSE=9.166950 epoch=  4 seconds=1.8
winsor0.005_s42_builtin                    MSE=84.109764 RMSE=9.171138 epoch=  4 seconds=1.7
winsor0.01_s42_builtin                     MSE=84.184716 RMSE=9.175223 epoch=  4 seconds=1.8
asinh0.25_s42_builtin                      MSE=86.722749 RMSE=9.312505 epoch=  2 seconds=1.6
asinh1_s42_builtin                     

## Подтверждение и финальные файлы

Ниже сохранены все проведённые этапы поиска, подтверждение на дополнительных разбиениях и воспроизводимое обучение трёх отдельных финальных сетей.


### Второй этап: ограниченное расширение признаков и взаимодействия

Первый этап показал лучшую MSE у PCA-поворота. Полный ортогональный поворот не добавляет информации: отличие может быть следствием инициализации и оптимизации, поэтому важны повторные разбиения. Дополнительно проверяются квадратичные признаки средних, квадраты всех признаков, нормирование групп каждой строки с сохранением масштаба, степенные преобразования и сочетание лучших преобразований с умеренно более сильным dropout. Максимальная размерность — 182, без комбинаторного взрыва по 90 признакам.


In [10]:
class ExtendedPreprocessor(SearchPreprocessor):
    def __init__(self, spec):
        self.original_spec = dict(spec)
        self.engineered = spec["kind"] == "engineered"
        super().__init__({"kind":"standard"} if self.engineered else spec)

    def engineer(self, z):
        name = self.original_spec["feature"]
        if name == "means_quad":
            a = np.clip(z[:,:12],-5,5); ii,jj=np.triu_indices(12)
            return np.concatenate([z,a[:,ii]*a[:,jj]],axis=1)
        if name == "means_sq": return np.concatenate([z,np.clip(z[:,:12],-5,5)**2],axis=1)
        if name == "all_sq": return np.concatenate([z,np.clip(z,-5,5)**2],axis=1)
        if name.startswith("power"):
            p=float(name.removeprefix("power"));return np.sign(z)*np.abs(z)**p
        a,b=z[:,:12],z[:,12:]
        na=np.maximum(np.sqrt(np.mean(a*a,axis=1,keepdims=True)),1e-5)
        nb=np.maximum(np.sqrt(np.mean(b*b,axis=1,keepdims=True)),1e-5)
        if name == "blocknorm":return np.concatenate([a/na,b/nb,np.log1p(na),np.log1p(nb)],axis=1)
        if name == "blocknorm_mix":return np.concatenate([z,a/na,b/nb,np.log1p(na),np.log1p(nb)],axis=1)
        if name == "covnorm":return np.concatenate([a,b/nb,np.log1p(nb)],axis=1)
        if name == "meansnorm":return np.concatenate([a/na,b,np.log1p(na)],axis=1)
        raise ValueError(name)

    def fit(self,x):
        super().fit(x)
        if self.engineered:
            z=super().transform(x).numpy()
            self.engineer_scaler=StandardScaler().fit(self.engineer(z))
        return self

    def transform(self,x):
        z=super().transform(x)
        if self.engineered:
            z=torch.from_numpy(self.engineer_scaler.transform(self.engineer(z.numpy())).astype(np.float32))
        assert torch.isfinite(z).all()
        return z

# Идентичный цикл обучения; только класс preprocessing заменён на расширение.


In [11]:
def run_extended(config, split_seed=42, max_epochs=80, patience=15, optimizer_kind="builtin"):
    ti, vi = train_test_split(development_idx, test_size=0.15/0.85, random_state=split_seed,
                             stratify=periods[development_idx])
    seed_everything(split_seed)
    pp = ExtendedPreprocessor(config["preprocessing"]).fit(X[ti])
    xt, xv = pp.transform(X[ti]), pp.transform(X[vi])
    scaler = TargetScaler(y[ti])
    loader = make_loader(xt, scaler.encode(y[ti]), split_seed)
    model = YearRegressor(xt.shape[1], config["architecture"], config["width"], config["dropout"])
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best_mse, stale, history, best_epoch, best_state = math.inf, 0, [], 0, None
    started = time.perf_counter()
    for epoch in range(1, max_epochs+1):
        for group in optimizer.param_groups: group["lr"] = learning_rate(config, epoch)
        loss, grad = train_epoch(model, loader, optimizer)
        pred = predict(model, xv, scaler)
        metrics = regression_metrics(pred, y[vi])
        history.append(dict(epoch=epoch, train_loss=loss, grad_norm=grad, **metrics))
        if metrics["mse"] < best_mse:
            best_mse, best_epoch, stale = metrics["mse"], epoch, 0
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: stale += 1
        if stale >= patience: break
    model.load_state_dict(best_state)
    pred = predict(model, xv, scaler)
    metrics = regression_metrics(pred, y[vi])
    rid = f"{config['id']}_s{split_seed}_{optimizer_kind}"
    result = dict(id=rid, config=config, split_seed=split_seed, best_epoch=best_epoch,
                  metrics=metrics, history=history, seconds=time.perf_counter()-started,
                  optimizer=optimizer_kind, checkpoint=str(ARTIFACT_DIR/f"{rid}.pt"))
    checkpoint = dict(state_dict=best_state, config=config, split_seed=split_seed,
                      target_mean=scaler.mean, target_std=scaler.std, train_idx=torch.tensor(ti),
                      valid_idx=torch.tensor(vi), valid_prediction=pred, result=result)
    torch.save(checkpoint, ARTIFACT_DIR/f"{rid}.pt")
    with open(ARTIFACT_DIR/f"{rid}_preprocessing.pkl", "wb") as f: pickle.dump(pp, f)
    print(f"{rid:42s} MSE={metrics['mse']:.6f} RMSE={metrics['rmse']:.6f} epoch={best_epoch:3d} seconds={result['seconds']:.1f}", flush=True)
    return result




In [12]:
stage2=[]
for feature in ["means_quad","means_sq","all_sq","blocknorm","blocknorm_mix","covnorm","meansnorm","power0.5","power0.75","power1.25"]:
    stage2.append(dict(common,id=feature,preprocessing=dict(kind="engineered",feature=feature)))
for pp_id,pp in [("standard",dict(kind="standard")),("pca0",dict(kind="pca",whiten_power=0.)),
                 ("pca0.5",dict(kind="pca",whiten_power=.5)),("rowscale",dict(kind="augment"))]:
    for width,dropout in [(64,.15),(128,.35),(64,.35)]:
        cfg=dict(common,id=f"{pp_id}_w{width}_d{dropout}",preprocessing=pp)
        cfg.update(width=width,dropout=dropout)
        stage2.append(cfg)
for config in stage2:
    result=run_extended(config)
    results.append(result);persist_results(results)
print("\nЛучшие результаты после второго этапа:")
for r in sorted(results,key=lambda r:r['metrics']['mse'])[:15]:
    print(r['config']['id'],r['metrics'],'epoch',r['best_epoch'])


means_quad_s42_builtin                     MSE=83.352713 RMSE=9.129771 epoch=  2 seconds=1.8
means_sq_s42_builtin                       MSE=85.104338 RMSE=9.225201 epoch=  3 seconds=1.7
all_sq_s42_builtin                         MSE=84.981317 RMSE=9.218531 epoch=  3 seconds=1.9
blocknorm_s42_builtin                      MSE=83.131707 RMSE=9.117659 epoch=  2 seconds=1.6
blocknorm_mix_s42_builtin                  MSE=83.553416 RMSE=9.140756 epoch=  2 seconds=1.7
covnorm_s42_builtin                        MSE=85.298358 RMSE=9.235711 epoch=  2 seconds=1.5
meansnorm_s42_builtin                      MSE=82.348238 RMSE=9.074593 epoch=  2 seconds=1.5
power0.5_s42_builtin                       MSE=88.098512 RMSE=9.386081 epoch=  2 seconds=1.6
power0.75_s42_builtin                      MSE=85.366507 RMSE=9.239400 epoch=  4 seconds=1.7
power1.25_s42_builtin                      MSE=83.956656 RMSE=9.162786 epoch=  3 seconds=1.7
standard_w64_d0.15_s42_builtin             MSE=83.593383 RMSE=9.142942

### Подтверждение четырёх заранее выбранных вариантов

Три лидирующие идеи первого этапа — PCA rotation, PCA partial-whitening и row-scale — сравниваются с исходным baseline на random_state 17 и 83. В каждом split пересоздаются train/validation внутри фиксированного development; audit исключён. Seed модели равен seed split. Эти splits пересекаются, поэтому значения нельзя трактовать как независимые доверительные интервалы.


In [13]:
confirmation_ids=["standard","pca0","pca0.5","rowscale"]
for cid in confirmation_ids:
    config=next(r['config'] for r in results if r['config']['id']==cid and r['split_seed']==42)
    for split_seed in [17,83]:
        result=run_extended(config,split_seed=split_seed,max_epochs=160,patience=25)
        results.append(result);persist_results(results)
print("\nПодтверждение на трёх разбиениях:")
for cid in confirmation_ids:
    rr=[r for r in results if r['config']['id']==cid and r['optimizer']=='builtin']
    print(cid, 'MSE=',[round(r['metrics']['mse'],5) for r in rr],
          'mean=',statistics.mean(r['metrics']['mse'] for r in rr),
          'epochs=',[r['best_epoch'] for r in rr])


standard_s17_builtin                       MSE=79.643057 RMSE=8.924296 epoch=  4 seconds=2.7
standard_s83_builtin                       MSE=87.147965 RMSE=9.335307 epoch=  3 seconds=2.5
pca0_s17_builtin                           MSE=79.103209 RMSE=8.893998 epoch=  4 seconds=2.7
pca0_s83_builtin                           MSE=85.349145 RMSE=9.238460 epoch=  3 seconds=2.5
pca0.5_s17_builtin                         MSE=79.516896 RMSE=8.917225 epoch=  3 seconds=2.5
pca0.5_s83_builtin                         MSE=86.179802 RMSE=9.283308 epoch=  3 seconds=2.6
rowscale_s17_builtin                       MSE=80.104545 RMSE=8.950114 epoch=  4 seconds=2.7
rowscale_s83_builtin                       MSE=86.101800 RMSE=9.279106 epoch=  3 seconds=2.5

Подтверждение на трёх разбиениях:
standard MSE= [83.88908, 79.64306, 87.14797] mean= 83.56003286571729 epochs= [4, 4, 3]
pca0 MSE= [81.89458, 79.10321, 85.34915] mean= 82.11564460152908 epochs= [4, 4, 3]
pca0.5 MSE= [82.3826, 79.5169, 86.1798] mean= 82.69

### Третий этап: взаимодействие с улучшенной BatchNorm-сетью

Параллельные исследования архитектуры выявили сильные ориентиры: два Linear/BatchNorm/SiLU/Dropout блока ширины 128 с dropout0.3, AdamW decay0.03; а также ReLU/dropout0.4/decay0.01. Проверяем, сохраняется ли выигрыш preprocessing уже с этими сильными моделями. Это одна сеть в каждом запуске; предсказания сетей не смешиваются.


In [14]:
class BatchNormRegressor(nn.Module):
    def __init__(self,n_features,architecture="bn_silu",width=128,dropout=.3):
        super().__init__()
        activation=nn.SiLU if architecture=="bn_silu" else nn.ReLU
        self.net=nn.Sequential(nn.Linear(n_features,width),nn.BatchNorm1d(width),activation(),nn.Dropout(dropout),
                               nn.Linear(width,width),nn.BatchNorm1d(width),activation(),nn.Dropout(dropout),nn.Linear(width,1))
    def forward(self,x):return self.net(x).squeeze(-1)


In [15]:
def run_bn(config, split_seed=42, max_epochs=80, patience=15, optimizer_kind="builtin"):
    ti, vi = train_test_split(development_idx, test_size=0.15/0.85, random_state=split_seed,
                             stratify=periods[development_idx])
    seed_everything(split_seed)
    pp = ExtendedPreprocessor(config["preprocessing"]).fit(X[ti])
    xt, xv = pp.transform(X[ti]), pp.transform(X[vi])
    scaler = TargetScaler(y[ti])
    loader = make_loader(xt, scaler.encode(y[ti]), split_seed)
    model = BatchNormRegressor(xt.shape[1], config["architecture"], config["width"], config["dropout"])
    optimizer = build_optimizer(model, optimizer_kind, config["lr"], config["weight_decay"])
    best_mse, stale, history, best_epoch, best_state = math.inf, 0, [], 0, None
    started = time.perf_counter()
    for epoch in range(1, max_epochs+1):
        for group in optimizer.param_groups: group["lr"] = learning_rate(config, epoch)
        loss, grad = train_epoch(model, loader, optimizer)
        pred = predict(model, xv, scaler)
        metrics = regression_metrics(pred, y[vi])
        history.append(dict(epoch=epoch, train_loss=loss, grad_norm=grad, **metrics))
        if metrics["mse"] < best_mse:
            best_mse, best_epoch, stale = metrics["mse"], epoch, 0
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else: stale += 1
        if stale >= patience: break
    model.load_state_dict(best_state)
    pred = predict(model, xv, scaler)
    metrics = regression_metrics(pred, y[vi])
    rid = f"{config['id']}_s{split_seed}_{optimizer_kind}"
    result = dict(id=rid, config=config, split_seed=split_seed, best_epoch=best_epoch,
                  metrics=metrics, history=history, seconds=time.perf_counter()-started,
                  optimizer=optimizer_kind, checkpoint=str(ARTIFACT_DIR/f"{rid}.pt"))
    checkpoint = dict(state_dict=best_state, config=config, split_seed=split_seed,
                      target_mean=scaler.mean, target_std=scaler.std, train_idx=torch.tensor(ti),
                      valid_idx=torch.tensor(vi), valid_prediction=pred, result=result)
    torch.save(checkpoint, ARTIFACT_DIR/f"{rid}.pt")
    with open(ARTIFACT_DIR/f"{rid}_preprocessing.pkl", "wb") as f: pickle.dump(pp, f)
    print(f"{rid:42s} MSE={metrics['mse']:.6f} RMSE={metrics['rmse']:.6f} epoch={best_epoch:3d} seconds={result['seconds']:.1f}", flush=True)
    return result




In [16]:
bn_configs=[]
pp_specs=[("standard",dict(kind="standard")),("pca0",dict(kind="pca",whiten_power=0)),
          ("pca0.25",dict(kind="pca",whiten_power=.25)),("pca0.5",dict(kind="pca",whiten_power=.5)),
          ("pca1",dict(kind="pca",whiten_power=1.)),("clip3",dict(kind="clip",limit=3.)),
          ("rowscale",dict(kind="augment")),("meansnorm",dict(kind="engineered",feature="meansnorm"))]
for arch,dropout,wd in [("bn_silu",.3,.03),("bn_relu",.4,.01)]:
    for ppid,spec in pp_specs:
        cfg=dict(common,id=f"{arch}_{ppid}",preprocessing=spec)
        cfg.update(architecture=arch,dropout=dropout,weight_decay=wd)
        bn_configs.append(cfg)
for cfg in bn_configs:
    result=run_bn(cfg,max_epochs=160,patience=25)
    results.append(result);persist_results(results)
print("\nBatchNorm и preprocessing:")
for r in sorted([r for r in results if r['config']['architecture'].startswith('bn_')],key=lambda r:r['metrics']['mse']):
    print(r['config']['id'],r['metrics'],'epoch',r['best_epoch'])


bn_silu_standard_s42_builtin               MSE=82.350821 RMSE=9.074735 epoch= 15 seconds=2.3
bn_silu_pca0_s42_builtin                   MSE=81.597166 RMSE=9.033115 epoch= 16 seconds=2.3
bn_silu_pca0.25_s42_builtin                MSE=81.784671 RMSE=9.043488 epoch= 16 seconds=2.3
bn_silu_pca0.5_s42_builtin                 MSE=82.119002 RMSE=9.061954 epoch= 16 seconds=2.3
bn_silu_pca1_s42_builtin                   MSE=82.764194 RMSE=9.097483 epoch= 14 seconds=2.2
bn_silu_clip3_s42_builtin                  MSE=82.221432 RMSE=9.067603 epoch= 13 seconds=2.2
bn_silu_rowscale_s42_builtin               MSE=81.728987 RMSE=9.040409 epoch= 18 seconds=2.5
bn_silu_meansnorm_s42_builtin              MSE=81.282611 RMSE=9.015687 epoch= 20 seconds=2.6
bn_relu_standard_s42_builtin               MSE=80.990101 RMSE=8.999450 epoch= 18 seconds=2.3
bn_relu_pca0_s42_builtin                   MSE=81.047836 RMSE=9.002657 epoch= 15 seconds=2.1
bn_relu_pca0.25_s42_builtin                MSE=81.353539 RMSE=9.019620

### Последняя волна: сильная регуляризация и подтверждение

Следующая общая архитектурная находка — BN/ReLU ширины 128, два блока, dropout0.55, weight decay0.5. Здесь сравниваем её с теми же представлениями входа. Предыдущие результаты показали, что выигрыш PCA в residual-сети почти не переносится на BatchNorm, поэтому дополнительно важны четыре row-scale признака и нормирование блока средних с сохранением его масштаба.

После этой волны две лучшие конфигурации по split42 фиксируются для подтверждения на 17/83. Дополнительно подтверждается лучший вариант предыдущей BN/ReLU волны. Новых настроек по результатам подтверждения или audit не добавляется.


In [17]:
strong_configs=[]
strong_specs=[("standard",dict(kind="standard")),("pca0",dict(kind="pca",whiten_power=0.)),
              ("pca0.25",dict(kind="pca",whiten_power=.25)),("clip3",dict(kind="clip",limit=3.)),
              ("rowscale",dict(kind="augment")),("meansnorm",dict(kind="engineered",feature="meansnorm")),
              ("blocknorm",dict(kind="engineered",feature="blocknorm")),("meansquad",dict(kind="engineered",feature="means_quad"))]
for pid,spec in strong_specs:
    cfg=dict(common,id=f"strong_{pid}",preprocessing=spec)
    cfg.update(architecture="bn_relu",dropout=.55,weight_decay=.5)
    strong_configs.append(cfg)
for cfg in strong_configs:
    result=run_bn(cfg,max_epochs=160,patience=25)
    results.append(result);persist_results(results)
strong_ranked=sorted([r for r in results if r['config']['id'].startswith('strong_')],key=lambda r:r['metrics']['mse'])
print('Сильная регуляризация, ранжирование:')
for r in strong_ranked:print(r['config']['id'],r['metrics'],'epoch',r['best_epoch'])
finalist_ids=list(dict.fromkeys([r['config']['id'] for r in strong_ranked[:2]]+['bn_relu_rowscale','strong_standard']))
print('Зафиксированы для подтверждения:',finalist_ids)
for cid in finalist_ids:
    cfg=next(r['config'] for r in results if r['config']['id']==cid)
    for seed in [17,83]:
        result=run_bn(cfg,split_seed=seed,max_epochs=160,patience=25)
        results.append(result);persist_results(results)
print('Подтверждение финалистов:')
for cid in finalist_ids:
    rr=[r for r in results if r['config']['id']==cid and r['optimizer']=='builtin']
    print(cid, 'MSE=',[round(r['metrics']['mse'],6) for r in rr],
          'mean=',statistics.mean(r['metrics']['mse'] for r in rr),
          'epochs=',[r['best_epoch'] for r in rr])


strong_standard_s42_builtin                MSE=80.916541 RMSE=8.995362 epoch= 35 seconds=2.9
strong_pca0_s42_builtin                    MSE=81.282583 RMSE=9.015685 epoch= 24 seconds=2.5
strong_pca0.25_s42_builtin                 MSE=81.776657 RMSE=9.043045 epoch= 29 seconds=2.8
strong_clip3_s42_builtin                   MSE=81.139487 RMSE=9.007746 epoch= 33 seconds=3.1
strong_rowscale_s42_builtin                MSE=80.557146 RMSE=8.975363 epoch= 18 seconds=2.2
strong_meansnorm_s42_builtin               MSE=80.555159 RMSE=8.975253 epoch= 45 seconds=3.7
strong_blocknorm_s42_builtin               MSE=81.654591 RMSE=9.036293 epoch= 29 seconds=2.9
strong_meansquad_s42_builtin               MSE=80.075491 RMSE=8.948491 epoch= 18 seconds=2.4
Сильная регуляризация, ранжирование:
strong_meansquad {'rmse': 8.948490971297462, 'mse': 80.0754906633922, 'mae': 6.2721519252232145} epoch 18
strong_meansnorm {'rmse': 8.97525259792518, 'mse': 80.55515919656271, 'mae': 6.305433233351935} epoch 45
strong_r

## Собственный оптимизатор и три финальных submission

Фиксируются три разные конфигурации с повторными проверками: квадратичные признаки средних + сильная регуляризация; нормирование блока средних + сильная регуляризация; четыре row-scale признака + умеренная регуляризация. Это три альтернативы для отдельных отправок, не ансамбль.

Самый низкий средний MSE здесь у квадратичных признаков —80.3822 против 80.4038 у сильного стандартного ориентира. Разница 0.0216 слишком мала для заявления о существенном улучшении preprocessing. Основной выигрыш относительно исходной сети 83.5600 связан с BatchNorm и регуляризацией. Split42 сам по себе переоценивает часть находок; public score этих кандидатов неизвестен.

Сначала повторяется лучший вариант с собственной реализацией AdamW при идентичных seed, батчах и dropout. Затем каждая финальная сеть учится на всех 14000 строках с собственным AdamW. Число эпох — медиана лучших эпох трёх split; горизонт cosine scheduler160 остаётся неизменным. Предсказания сохраняются в `task4_data/Submission`, порядок 6000test строк и схема`id,year` строго сохраняются. Audit не используется для оценки или выбора в этом исследовании.


In [18]:
submission_ids=["strong_meansquad","strong_meansnorm","bn_relu_rowscale"]
validated_summary=[]
for cid in submission_ids+["strong_standard","standard","pca0"]:
    rr=[r for r in results if r['config']['id']==cid and r['optimizer']=='builtin']
    assert {r['split_seed'] for r in rr}=={42,17,83}
    validated_summary.append(dict(id=cid,mean_mse=statistics.mean(r['metrics']['mse'] for r in rr),
                       mse_by_seed={str(r['split_seed']):r['metrics']['mse'] for r in rr},
                       refit_epochs=int(statistics.median(r['best_epoch'] for r in rr))))
for row in sorted(validated_summary,key=lambda r:r['mean_mse']):print(row)

best_config=next(r['config'] for r in results if r['config']['id']==submission_ids[0])
custom_result=run_bn(best_config,max_epochs=160,patience=25,optimizer_kind="custom")
results.append(custom_result);persist_results(results)
builtin_result=next(r for r in results if r['config']['id']==submission_ids[0] and r['split_seed']==42 and r['optimizer']=='builtin')
a=torch.load(builtin_result['checkpoint'],map_location='cpu',weights_only=False)
b=torch.load(custom_result['checkpoint'],map_location='cpu',weights_only=False)
assert torch.equal(a['valid_idx'],b['valid_idx'])
prediction_difference=(a['valid_prediction']-b['valid_prediction']).abs().max().item()
print('Custom minus builtin MSE:',custom_result['metrics']['mse']-builtin_result['metrics']['mse'])
print('Maximum absolute prediction difference:',prediction_difference)
assert prediction_difference < 1e-4


{'id': 'strong_meansquad', 'mean_mse': 80.38218508722763, 'mse_by_seed': {'42': 80.0754906633922, '17': 77.6718834337734, '83': 83.39918116451729}, 'refit_epochs': 18}
{'id': 'strong_standard', 'mean_mse': 80.4038068695319, 'mse_by_seed': {'42': 80.91654095286415, '17': 76.32170434769421, '83': 83.97317530803737}, 'refit_epochs': 31}
{'id': 'strong_meansnorm', 'mean_mse': 80.47002465148768, 'mse_by_seed': {'42': 80.55515919656271, '17': 77.12774134840994, '83': 83.72717340949036}, 'refit_epochs': 25}
{'id': 'bn_relu_rowscale', 'mean_mse': 80.68002337524106, 'mse_by_seed': {'42': 80.59419161503514, '17': 77.01586134371303, '83': 84.430017166975}, 'refit_epochs': 19}
{'id': 'pca0', 'mean_mse': 82.11564460152908, 'mse_by_seed': {'42': 81.89457985551584, '17': 79.10320862580623, '83': 85.34914532326516}, 'refit_epochs': 4}
{'id': 'standard', 'mean_mse': 83.56003286571729, 'mse_by_seed': {'42': 83.88907607429084, '17': 79.64305718292083, '83': 87.14796533994021}, 'refit_epochs': 4}
strong_m

In [19]:
SUBMISSION_DIR=DATA_DIR/'Submission'
SUBMISSION_DIR.mkdir(exist_ok=True)
final_records=[]
for cid in submission_ids:
    config=next(r['config'] for r in results if r['config']['id']==cid)
    epochs=next(r['refit_epochs'] for r in validated_summary if r['id']==cid)
    seed_everything(42)
    pp=ExtendedPreprocessor(config['preprocessing']).fit(X)
    xt=pp.transform(X)
    scaler=TargetScaler(y)
    loader=make_loader(xt,scaler.encode(y),42)
    model=BatchNormRegressor(xt.shape[1],config['architecture'],config['width'],config['dropout'])
    optimizer=build_optimizer(model,'custom',config['lr'],config['weight_decay'])
    losses=[]
    for epoch in range(1,epochs+1):
        for group in optimizer.param_groups:group['lr']=learning_rate(config,epoch)
        loss,gradient=train_epoch(model,loader,optimizer)
        losses.append(dict(epoch=epoch,loss=loss,grad_norm=gradient))
    prediction=predict(model,pp.transform(X_test),scaler)
    assert prediction.shape==(6000,) and torch.isfinite(prediction).all()
    submission=pd.DataFrame({'id':test_ids.to_numpy(),'year':prediction.numpy()})
    output_path=SUBMISSION_DIR/f'task4_preprocessing_{cid}.csv'
    submission.to_csv(output_path,index=False,float_format='%.6f')
    reloaded=pd.read_csv(output_path)
    assert list(reloaded.columns)==['id','year'] and len(reloaded)==6000
    assert reloaded['id'].equals(test_ids.reset_index(drop=True))
    torch.testing.assert_close(torch.tensor(reloaded['year'].to_numpy()),prediction.double(),rtol=0,atol=1e-6)
    checkpoint=ARTIFACT_DIR/f'final_{cid}.pt'
    torch.save(dict(state_dict=model.state_dict(),config=config,target_mean=scaler.mean,
                    target_std=scaler.std,epochs=epochs,history=losses,test_prediction=prediction),checkpoint)
    with open(ARTIFACT_DIR/f'final_{cid}_preprocessing.pkl','wb') as f:pickle.dump(pp,f)
    record=dict(id=cid,config=config,epochs=epochs,csv=str(output_path),checkpoint=str(checkpoint),
                prediction_min=float(prediction.min()),prediction_max=float(prediction.max()),
                sha256=hashlib.sha256(output_path.read_bytes()).hexdigest())
    final_records.append(record)
    print(json.dumps(record,ensure_ascii=False))

# Structural checks: every reported validation row is outside the reserved audit pool.
for result in results:
    checkpoint=torch.load(result['checkpoint'],map_location='cpu',weights_only=False)
    train_set,valid_set=set(checkpoint['train_idx'].tolist()),set(checkpoint['valid_idx'].tolist())
    assert len(train_set)==9800 and len(valid_set)==2100
    assert not train_set & valid_set
    assert not (train_set|valid_set) & set(audit_idx.tolist())

baseline_path=Path('homeworks/task4.ipynb') if Path('homeworks/task4.ipynb').is_file() else Path('task4.ipynb')
assert hashlib.sha256(baseline_path.read_bytes()).hexdigest()=='d3883b0f7909703c335dcdd44e252ef8101d0fea16b3b0da1fa3b33807735419'
print('Исходный task4.ipynb неизменён. Audit не оценивался. CSV полностью проверены.')
print('Уникальных конфигураций:',len({r['config']['id'] for r in results}),'запусков:',len(results))
final_report=dict(search_and_confirmation=results,comparison=validated_summary,submissions=final_records,
                  audit_evaluated=False,custom_max_prediction_difference=prediction_difference)
(ARTIFACT_DIR/'final_report.json').write_text(json.dumps(final_report,indent=2))


{"id": "strong_meansquad", "config": {"architecture": "bn_relu", "width": 128, "dropout": 0.55, "lr": 0.001, "weight_decay": 0.5, "id": "strong_meansquad", "preprocessing": {"kind": "engineered", "feature": "means_quad"}}, "epochs": 18, "csv": "homeworks/task4_data/Submission/task4_preprocessing_strong_meansquad.csv", "checkpoint": "/tmp/task4_preprocessing/final_strong_meansquad.pt", "prediction_min": 1972.7177734375, "prediction_max": 2006.18212890625, "sha256": "aa56eb7f3015d371bb59bac995577a00dd562447621fc76a4e21b4a11a212497"}
{"id": "strong_meansnorm", "config": {"architecture": "bn_relu", "width": 128, "dropout": 0.55, "lr": 0.001, "weight_decay": 0.5, "id": "strong_meansnorm", "preprocessing": {"kind": "engineered", "feature": "meansnorm"}}, "epochs": 25, "csv": "homeworks/task4_data/Submission/task4_preprocessing_strong_meansnorm.csv", "checkpoint": "/tmp/task4_preprocessing/final_strong_meansnorm.pt", "prediction_min": 1978.1881103515625, "prediction_max": 2005.91845703125, "s

## Выводы и воспроизводимость

- Проверено 76 уникальных конфигураций:30 преобразований при фиксированной сети,22 варианта добавленных признаков и взаимодействий,16 сочетаний с BatchNorm-сетями и 8 сочетаний с сильной регуляризацией. Лучшие гипотезы повторены на splits17/83; итоговые числа и эпохи напечатаны выше.
- Сильное asinh/log/quantile сжатие обычно ухудшало качество исходной residual-сети. PCA помог ей, но не дал систематического выигрыша уже регуляризованной BatchNorm-сети. Это пример зависимости полезности preprocessing от архитектуры.
- Средний MSE трёх разбиений у сильной стандартной BN-сети 80.4038, у её варианта с квадратичными признаками 80.3822. Существенное преимущество именно инженерии признаков не установлено. Сравнение с исходным 83.5600 показывает общий прогресс, но не гарантирует улучшения public/private leaderboard.
- Разбиения пересекаются, а результаты использовались для выбора; среднее по ним не является независимой оценкой скрытого теста. До фиксации кандидатов audit не открывался. Для окончательного решения нужна зафиксированная независимая проверка и результаты отправок.
- Каждый CSV получен одной сетью. Собственный AdamW проверен тестами и сравнением с PyTorch; ансамбли, внешние данные и идентификаторы как признаки не используются.
- Для воспроизведения достаточно этого ноутбука, трёх исходных CSV и стандартных установленных библиотек. `/tmp` используется только для сохранения результатов текущего запуска и не является входом решения.

Источники: локальные `homeworks/pytorch.md`, `notebooks/pytorch_intro.ipynb`, исходное решение `task4.ipynb`; алгоритм собственного AdamW — [Loshchilov & Hutter](https://arxiv.org/abs/1711.05101), API — [PyTorch AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html). Внешние данные не загружались.


## Приложение: однократная диагностика на audit после фиксации всех решений

Все семейства, гиперпараметры, shortlist и число эпох финального обучения зафиксированы до этой проверки. Audit2100 строк проверяется на checkpoint seed42, обученном только на 9800train строках, а не на модели после final refit. Никакие параметры и submission после просмотра этих чисел не меняются.

Audit исходного baseline уже был показан в предыдущем решении. В данном поиске он не использовался для ранней остановки или настройки. Поэтому это дополнительная диагностическая выборка, но не полностью ранее невиданный внешний тест. Главный кандидат `combined_meansquad` выбран по трём development-разбиениям и остаётся выбранным независимо от audit.


In [20]:
audit_records=[];audit_predictions={}
for cid in ['strong_meansquad','strong_meansnorm','bn_relu_rowscale','standard']:
    prefix=ARTIFACT_DIR/f'{cid}_s42_builtin'
    cp=torch.load(str(prefix)+'.pt',weights_only=False,map_location='cpu')
    with open(str(prefix)+'_preprocessing.pkl','rb') as f:pp=pickle.load(f)
    config=cp['config']
    if config['architecture'].startswith('bn_'):
        model=BatchNormRegressor(pp.transform(X[cp['train_idx'][:1].tolist()]).shape[1],config['architecture'],config['width'],config['dropout'])
    else:
        model=YearRegressor(90,config['architecture'],config['width'],config['dropout'])
    model.load_state_dict(cp['state_dict'])
    scaler=TargetScaler(y[cp['train_idx']]);scaler.mean=cp['target_mean'];scaler.std=cp['target_std']
    prediction=predict(model,pp.transform(X[audit_idx]),scaler)
    metric=regression_metrics(prediction,y[audit_idx])
    record=dict(id=cid,config=config,seed=42,training_rows=9800,audit_rows=2100,metrics=metric)
    audit_records.append(record);audit_predictions[cid]=prediction
    print(json.dumps(record,ensure_ascii=False))
Path('/tmp/task4_preprocessing_audit.json').write_text(json.dumps(audit_records,indent=2))
torch.save(dict(audit_idx=torch.tensor(audit_idx),predictions=audit_predictions,records=audit_records),'/tmp/task4_preprocessing_audit_predictions.pt')
report=json.loads((ARTIFACT_DIR/'final_report.json').read_text())
report.update(audit_evaluated=True,audit_diagnostic=audit_records,audit_used_for_selection=False)
(ARTIFACT_DIR/'final_report.json').write_text(json.dumps(report,indent=2))


{"id": "strong_meansquad", "config": {"architecture": "bn_relu", "width": 128, "dropout": 0.55, "lr": 0.001, "weight_decay": 0.5, "id": "strong_meansquad", "preprocessing": {"kind": "engineered", "feature": "means_quad"}}, "seed": 42, "training_rows": 9800, "audit_rows": 2100, "metrics": {"rmse": 9.088232398167793, "mse": 82.59596812310673, "mae": 6.370452357700893}}
{"id": "strong_meansnorm", "config": {"architecture": "bn_relu", "width": 128, "dropout": 0.55, "lr": 0.001, "weight_decay": 0.5, "id": "strong_meansnorm", "preprocessing": {"kind": "engineered", "feature": "meansnorm"}}, "seed": 42, "training_rows": 9800, "audit_rows": 2100, "metrics": {"rmse": 9.011342683000732, "mse": 81.20429695047083, "mae": 6.311238141741072}}
{"id": "bn_relu_rowscale", "config": {"architecture": "bn_relu", "width": 128, "dropout": 0.4, "lr": 0.001, "weight_decay": 0.01, "id": "bn_relu_rowscale", "preprocessing": {"kind": "augment"}}, "seed": 42, "training_rows": 9800, "audit_rows": 2100, "metrics": 